In [1]:
import json
import os
import gc
import shutil
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable: enable a Kaggle GPU accelerator before running.")
if torch.cuda.device_count() < 2:
    raise RuntimeError("This notebook expects Kaggle T4 x2 (GPU 0 for training and GPU 1 for reference caching).")

inputroot = Path("/kaggle/input")
checkpointpaths = []
for configurationpath in inputroot.rglob("config.json"):
    candidate = configurationpath.parent
    hasweights = (candidate / "model.safetensors").is_file() or (candidate / "model.safetensors.index.json").is_file()
    if hasweights and (candidate / "tokenizer_config.json").is_file() and (candidate / "checkpoint.json").is_file():
        checkpointpaths.append(candidate)

if len(checkpointpaths) != 1:
    raise RuntimeError(f"Expected exactly one original SFT checkpoint with checkpoint.json, found {len(checkpointpaths)}: {checkpointpaths}. Attach only the original SFT model dataset or set checkpointpath explicitly.")
checkpointpath = checkpointpaths[0]
checkpointmetadata = json.loads((checkpointpath / "checkpoint.json").read_text(encoding="utf-8"))
print("SFT checkpoint:", checkpointpath)
print("CUDA devices:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("Checkpoint metadata:", json.dumps(checkpointmetadata, indent=2))


SFT checkpoint: /kaggle/input/datasets/kangerz22/pa2sftcheckpoint
CUDA devices: ['Tesla T4', 'Tesla T4']
Checkpoint metadata: {
  "optimizer_step": 938,
  "validation_loss": 1.3795008834900906,
  "selection": "lowest validation loss"
}


In [3]:
import os
import subprocess
import tempfile
from pathlib import Path

from kaggle_secrets import UserSecretsClient

repositoryurl = "https://github.com/tachytach1/pa2-small-model-alignment.git"
repositorypath = Path("/kaggle/working/pa2-small-model-alignment")

if repositorypath.exists():
    print("Using existing repository:", repositorypath)
else:
    githubtoken = UserSecretsClient().get_secret("GITHUB_TOKEN")

    with tempfile.TemporaryDirectory() as temporarydirectory:
        helperpath = Path(temporarydirectory) / "git-askpass.sh"

        helperpath.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
            '  *Password*) printf "%s\\n" "$KAGGLE_GITHUB_TOKEN" ;;\n'
            'esac\n',
            encoding="utf-8",
        )
        helperpath.chmod(0o700)

        environment = os.environ.copy()
        environment["GIT_ASKPASS"] = str(helperpath)
        environment["GIT_TERMINAL_PROMPT"] = "0"
        environment["KAGGLE_GITHUB_TOKEN"] = githubtoken

        try:
            subprocess.run(
                [
                    "git",
                    "-c",
                    "credential.helper=",
                    "clone",
                    repositoryurl,
                    str(repositorypath),
                ],
                env=environment,
                check=True,
            )
        finally:
            environment.pop("KAGGLE_GITHUB_TOKEN", None)
            del githubtoken

Cloning into '/kaggle/working/pa2-small-model-alignment'...


In [4]:
%cd /kaggle/working/pa2-small-model-alignment
!python -m pip uninstall -y -q torchao
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm matplotlib
!python -m pip install -q -e .

import torch
import matplotlib.pyplot as plt
from pathlib import Path
print("Repository and dependencies ready.")


/kaggle/working/pa2-small-model-alignment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 103.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building edit

In [5]:
runhh = True
runuf = True
runlrsweep = False  # optional extra experiments; see last section

hhlearningrate = 2e-6
uflearningrate = 2e-6
hhepochs = 3
ufepochs = 2
beta = 0.1
accumulation = 32
checkpointinterval = 200
validationinterval = 100
seed = 42
hhmaximumlength = 512
ufmaximumlength = 512
maximumpromptlength = 256
trainingpairs = 15000
validationpairs = 1000

outputroot = Path("outputs/dpo_retrain_2026")
outputroot.mkdir(parents=True, exist_ok=True)

tracksettings = {
    "hh": {
        "prepared": "outputs/preferences_hh_full",
        "cache": "outputs/reference_hh_full",
        "learningrate": hhlearningrate,
        "epochs": hhepochs,
        "maximumlength": hhmaximumlength,
        "revision": "09be8c5bbc57cb3887f3a9732ad6aa7ec602a1fa",
        "enabled": runhh,
    },
    "ultrafeedback": {
        "prepared": "outputs/preferences_ultrafeedback_full",
        "cache": "outputs/reference_ultrafeedback_full",
        "learningrate": uflearningrate,
        "epochs": ufepochs,
        "maximumlength": ufmaximumlength,
        "revision": "3949bf5f8c17c394422ccfab0c31ea9c20bdeb85",
        "enabled": runuf,
    },
}

print(json.dumps(tracksettings, indent=2))
print("Free working disk (GiB):", round(shutil.disk_usage("/kaggle/working").free / 1024**3, 2))


{
  "hh": {
    "prepared": "outputs/preferences_hh_full",
    "cache": "outputs/reference_hh_full",
    "learningrate": 2e-06,
    "epochs": 3,
    "maximumlength": 512,
    "revision": "09be8c5bbc57cb3887f3a9732ad6aa7ec602a1fa",
    "enabled": true
  },
  "ultrafeedback": {
    "prepared": "outputs/preferences_ultrafeedback_full",
    "cache": "outputs/reference_ultrafeedback_full",
    "learningrate": 2e-06,
    "epochs": 2,
    "maximumlength": 512,
    "revision": "3949bf5f8c17c394422ccfab0c31ea9c20bdeb85",
    "enabled": true
  }
}
Free working disk (GiB): 19.5


In [6]:
%%writefile alignment/preferences.py
from __future__ import annotations

import hashlib
import json
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import HfApi
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from alignment.batching import encode_prompt_response, pad_encoded_examples
from alignment.data import formatinstruction


def extracthhpair(sourcerow):
    chosenconversation = sourcerow.get("chosen")
    rejectedconversation = sourcerow.get("rejected")

    if not isinstance(chosenconversation, str):
        return None

    if not isinstance(rejectedconversation, str):
        return None

    conversations = []

    for conversation in (chosenconversation, rejectedconversation):
        conversation = conversation.strip()

        if not conversation.startswith("Human:"):
            return None

        if conversation.count("\n\nHuman:") != 0:
            return None

        if conversation.count("\n\nAssistant:") != 1:
            return None

        instruction, response = conversation.split("\n\nAssistant:", 1)
        instruction = instruction.removeprefix("Human:").strip()
        response = response.strip()

        if not instruction or not response:
            return None

        conversations.append((instruction, response))

    choseninstruction, chosenresponse = conversations[0]
    rejectedinstruction, rejectedresponse = conversations[1]

    if choseninstruction != rejectedinstruction:
        return None

    return choseninstruction, chosenresponse, rejectedresponse


def extractultrafeedbackpair(sourcerow):
    chosenmessages = sourcerow.get("chosen")
    rejectedmessages = sourcerow.get("rejected")

    exchanges = []

    for messages in (chosenmessages, rejectedmessages):
        if not isinstance(messages, list) or len(messages) != 2:
            return None

        usermessage, assistantmessage = messages

        if not isinstance(usermessage, dict):
            return None

        if not isinstance(assistantmessage, dict):
            return None

        if usermessage.get("role") != "user":
            return None

        if assistantmessage.get("role") != "assistant":
            return None

        instruction = usermessage.get("content")
        response = assistantmessage.get("content")

        if not isinstance(instruction, str) or not isinstance(response, str):
            return None

        instruction = instruction.strip()
        response = response.strip()

        if not instruction or not response:
            return None

        exchanges.append((instruction, response))

    choseninstruction, chosenresponse = exchanges[0]
    rejectedinstruction, rejectedresponse = exchanges[1]

    if choseninstruction != rejectedinstruction:
        return None

    return choseninstruction, chosenresponse, rejectedresponse


def collatepreferencebatch(examples, paddingtokenid):
    chosenexamples = []
    rejectedexamples = []

    for example in examples:
        chosenexamples.append(example["chosen"])
        rejectedexamples.append(example["rejected"])

    return {
        "chosen": pad_encoded_examples(chosenexamples, paddingtokenid),
        "rejected": pad_encoded_examples(rejectedexamples, paddingtokenid),
    }


def preparepreferencesplit(
    tokenizer,
    datasetname,
    datasetrevision,
    sourcesplit,
    extractor,
    requestedpairs,
    maximumlength,
    maximumpromptlength,
    seed,
    shufflebuffer,
    seeninstructions,
):
    sourcedataset = load_dataset(
        datasetname,
        revision=datasetrevision,
        split=sourcesplit,
        streaming=True,
    ).shuffle(seed=seed, buffer_size=shufflebuffer)

    acceptedpairs = []

    statistics = {
        "source_split": sourcesplit,
        "examined_pairs": 0,
        "unsupported_or_invalid_pairs": 0,
        "duplicate_instructions": 0,
        "identical_response_pairs": 0,
        "long_prompt_pairs": 0,
        "pairs_without_response_tokens": 0,
        "identical_tokenized_pairs": 0,
        "accepted_pairs": 0,
        "chosen_response_tokens": 0,
        "rejected_response_tokens": 0,
    }

    with tqdm(
        total=requestedpairs,
        desc=f"Preparing {sourcesplit}",
        unit="pair",
    ) as progress:
        for sourcerow in sourcedataset:
            statistics["examined_pairs"] += 1
            extractedpair = extractor(sourcerow)

            if extractedpair is None:
                statistics["unsupported_or_invalid_pairs"] += 1
                continue

            instruction, chosenresponse, rejectedresponse = extractedpair

            instructionhash = hashlib.sha256(
                instruction.encode("utf-8")
            ).hexdigest()

            if instructionhash in seeninstructions:
                statistics["duplicate_instructions"] += 1
                continue

            if chosenresponse == rejectedresponse:
                statistics["identical_response_pairs"] += 1
                continue

            prompt = formatinstruction(instruction)
            promptlength = len(
                tokenizer.encode(prompt, add_special_tokens=False)
            )

            if promptlength > maximumpromptlength:
                statistics["long_prompt_pairs"] += 1
                continue

            chosenexample = encode_prompt_response(
                tokenizer,
                prompt,
                chosenresponse,
                maximumlength,
            )
            rejectedexample = encode_prompt_response(
                tokenizer,
                prompt,
                rejectedresponse,
                maximumlength,
            )

            chosenresponsecount = 0
            rejectedresponsecount = 0

            for tokenlabel in chosenexample["labels"][1:]:
                if tokenlabel != -100:
                    chosenresponsecount += 1

            for tokenlabel in rejectedexample["labels"][1:]:
                if tokenlabel != -100:
                    rejectedresponsecount += 1

            if chosenresponsecount == 0 or rejectedresponsecount == 0:
                statistics["pairs_without_response_tokens"] += 1
                continue

            if chosenexample["input_ids"] == rejectedexample["input_ids"]:
                statistics["identical_tokenized_pairs"] += 1
                continue

            acceptedpairs.append(
                {
                    "instruction_hash": instructionhash,
                    "instruction": instruction,
                    "prompt": prompt,
                    "chosen_response": chosenresponse,
                    "rejected_response": rejectedresponse,
                    "chosen": chosenexample,
                    "rejected": rejectedexample,
                    "chosen_response_token_count": chosenresponsecount,
                    "rejected_response_token_count": rejectedresponsecount,
                }
            )

            seeninstructions.add(instructionhash)
            statistics["accepted_pairs"] += 1
            statistics["chosen_response_tokens"] += chosenresponsecount
            statistics["rejected_response_tokens"] += rejectedresponsecount

            progress.update(1)

            if len(acceptedpairs) == requestedpairs:
                break

    if len(acceptedpairs) != requestedpairs:
        raise RuntimeError(
            f"{sourcesplit} supplied {len(acceptedpairs)} usable pairs, "
            f"but {requestedpairs} were requested. "
            f"Filtering statistics: {statistics}"
        )

    return Dataset.from_list(acceptedpairs), statistics


def preparepreferences(
    checkpointpath,
    track,
    outputpath,
    trainingpairs=15000,
    validationpairs=1000,
    maximumlength=512,
    maximumpromptlength=256,
    seed=42,
    shufflebuffer=1000,
    datasetrevision="main",
):
    for argumentname, argumentvalue in {
        "trainingpairs": trainingpairs,
        "validationpairs": validationpairs,
        "maximumlength": maximumlength,
        "maximumpromptlength": maximumpromptlength,
        "shufflebuffer": shufflebuffer,
    }.items():
        if argumentvalue < 1:
            raise ValueError(f"{argumentname} must be positive.")

    if maximumpromptlength >= maximumlength:
        raise ValueError("The prompt limit must leave room for response tokens.")

    if track == "hh":
        datasetname = "Anthropic/hh-rlhf"
        trainingsplit = "train"
        validationsplit = "test"
        extractor = extracthhpair
    elif track == "ultrafeedback":
        datasetname = "HuggingFaceH4/ultrafeedback_binarized"
        trainingsplit = "train_prefs"
        validationsplit = "test_prefs"
        extractor = extractultrafeedbackpair
    else:
        raise ValueError("track must be hh or ultrafeedback.")

    checkpointpath = Path(checkpointpath)
    destination = Path(outputpath)

    if destination.exists():
        raise FileExistsError(f"Output directory already exists: {destination}")

    tokenizer = AutoTokenizer.from_pretrained(
        str(checkpointpath),
        local_files_only=True,
        fix_mistral_regex=False,
    )

    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is None:
            raise ValueError("The tokenizer needs a padding or EOS token.")
        tokenizer.pad_token = tokenizer.eos_token

    resolvedrevision = HfApi().dataset_info(
        datasetname,
        revision=datasetrevision,
    ).sha

    seeninstructions = set()

    trainingdataset, trainingstatistics = preparepreferencesplit(
        tokenizer,
        datasetname,
        resolvedrevision,
        trainingsplit,
        extractor,
        trainingpairs,
        maximumlength,
        maximumpromptlength,
        seed,
        shufflebuffer,
        seeninstructions,
    )

    validationdataset, validationstatistics = preparepreferencesplit(
        tokenizer,
        datasetname,
        resolvedrevision,
        validationsplit,
        extractor,
        validationpairs,
        maximumlength,
        maximumpromptlength,
        seed,
        shufflebuffer,
        seeninstructions,
    )

    manifest = {
        "track": track,
        "dataset_name": datasetname,
        "dataset_revision": resolvedrevision,
        "sft_checkpoint": str(checkpointpath),
        "checkpoint_metadata": json.loads(
            (checkpointpath / "checkpoint.json").read_text(encoding="utf-8")
        ),
        "tokenizer_file_sha256": hashlib.sha256(
            (checkpointpath / "tokenizer.json").read_bytes()
        ).hexdigest(),
        "seed": seed,
        "shuffle_buffer": shufflebuffer,
        "maximum_sequence_length": maximumlength,
        "maximum_prompt_length": maximumpromptlength,
        "long_prompt_policy": "reject; do not truncate the instruction separately",
        "conversation_policy": "single user and assistant exchange only",
        "prompt_template": "### Instruction:\n{instruction}\n\n### Response:\n",
        "padding_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "training": trainingstatistics,
        "validation": validationstatistics,
    }

    destination.mkdir(parents=True, exist_ok=False)

    DatasetDict(
        {
            "train": trainingdataset,
            "validation": validationdataset,
        }
    ).save_to_disk(str(destination / "dataset"))

    (destination / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(json.dumps(manifest, indent=2))
    print("Prepared preference data:", destination.resolve())

    return manifest

Writing alignment/preferences.py


In [8]:
import gc
import json
import time
from functools import partial
from pathlib import Path

import torch

from datasets import DatasetDict, load_from_disk
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

from alignment.preferences import collatepreferencebatch
from alignment.scoring import compute_batch_response_statistics


def cachereferencescores(
    referencemodel,
    preparedpath,
    outputpath,
    batchsize=1,
):
    preparedpath = Path(preparedpath)
    outputpath = Path(outputpath)

    if outputpath.exists():
        raise FileExistsError(
            f"Reference cache already exists: {outputpath}"
        )

    preparationmanifest = json.loads(
        (preparedpath / "manifest.json").read_text(encoding="utf-8")
    )

    if preparationmanifest["checkpoint_metadata"] != checkpointmetadata:
        raise ValueError("Prepared data refers to a different SFT checkpoint.")

    prepareddataset = load_from_disk(
        str(preparedpath / "dataset")
    )

    collator = partial(
        collatepreferencebatch,
        paddingtokenid=preparationmanifest["padding_token_id"],
    )

    cacheddatasets = {}
    starttime = time.perf_counter()
    referencedevice = next(referencemodel.parameters()).device

    for splitname in ("train", "validation"):
        splitdataset = prepareddataset[splitname]

        loader = DataLoader(
            splitdataset,
            batch_size=batchsize,
            shuffle=False,
            collate_fn=collator,
            num_workers=0,
            pin_memory=True,
        )

        chosenscores = []
        rejectedscores = []

        with torch.inference_mode():
            for preferencebatch in tqdm(
                loader,
                desc=f"Reference scores: {splitname}",
            ):
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    chosenstatistics = compute_batch_response_statistics(
                        referencemodel,
                        preferencebatch["chosen"],
                    )

                    rejectedstatistics = compute_batch_response_statistics(
                        referencemodel,
                        preferencebatch["rejected"],
                    )

                chosenprobabilities = chosenstatistics[0]
                rejectedprobabilities = rejectedstatistics[0]

                if not torch.isfinite(chosenprobabilities).all().item():
                    raise FloatingPointError(
                        "Chosen reference scores are not finite."
                    )

                if not torch.isfinite(rejectedprobabilities).all().item():
                    raise FloatingPointError(
                        "Rejected reference scores are not finite."
                    )

                if (
                    chosenstatistics[1].eq(0).any().item()
                    or rejectedstatistics[1].eq(0).any().item()
                ):
                    raise ValueError(
                        "A preference response has no scoreable tokens."
                    )

                chosenscores.extend(
                    chosenprobabilities.float().cpu().tolist()
                )
                rejectedscores.extend(
                    rejectedprobabilities.float().cpu().tolist()
                )

        if (
            len(chosenscores) != len(splitdataset)
            or len(rejectedscores) != len(splitdataset)
        ):
            raise RuntimeError("Reference scores do not match dataset rows.")

        cacheddataset = splitdataset.add_column(
            "reference_chosen_logps",
            chosenscores,
        )

        cacheddataset = cacheddataset.add_column(
            "reference_rejected_logps",
            rejectedscores,
        )

        cacheddatasets[splitname] = cacheddataset

    cachemanifest = {
        "preparation": preparationmanifest,
        "reference_checkpoint": str(checkpointpath),
        "reference_checkpoint_metadata": checkpointmetadata,
        "reference_weight_dtype": str(
            next(referencemodel.parameters()).dtype
        ),
        "reference_autocast_dtype": "torch.float16",
        "reference_device": str(referencedevice),
        "score_reduction": "summed response-token log probabilities",
        "batch_size": batchsize,
        "elapsed_seconds": time.perf_counter() - starttime,
        "package_versions": {
            "torch": torch.__version__,
        },
    }

    outputpath.mkdir(parents=True, exist_ok=False)

    DatasetDict(cacheddatasets).save_to_disk(
        str(outputpath / "dataset")
    )

    (outputpath / "manifest.json").write_text(
        json.dumps(cachemanifest, indent=2),
        encoding="utf-8",
    )

    print("Reference cache saved:", outputpath.resolve())
    print(
        "Reference scoring time:",
        f"{cachemanifest['elapsed_seconds']:.2f} seconds",
    )

    return cachemanifest

In [9]:
from datasets import load_from_disk
from transformers import AutoModelForCausalLM

def locateexistingartifact(relativepath):
    target = Path(relativepath)
    if (target / "manifest.json").is_file() and (target / "dataset").is_dir():
        return target
    matches = []
    for manifestpath in Path("/kaggle/input").rglob("manifest.json"):
        if manifestpath.parent.name == target.name and (manifestpath.parent / "dataset").is_dir():
            matches.append(manifestpath.parent)
    if len(matches) > 1:
        raise RuntimeError(f"Ambiguous attached artifacts for {target.name}: {matches}. Attach only one matching dataset.")
    return matches[0] if matches else None

def checkprepared(preparedpath, track, maximumlength):
    manifest = json.loads((preparedpath / "manifest.json").read_text(encoding="utf-8"))
    if manifest.get("checkpoint_metadata") != checkpointmetadata:
        raise RuntimeError(f"Prepared {track} dataset was made from a different SFT checkpoint: {preparedpath}")
    if manifest.get("maximum_sequence_length") != maximumlength:
        print("Preparation manifest:", json.dumps(manifest, indent=2)[:3500])
        raise RuntimeError(f"Prepared {track} dataset token limit does not match {maximumlength}: {preparedpath}")
    return manifest

def checkcache(cachepath, track, maximumlength):
    manifest = json.loads((cachepath / "manifest.json").read_text(encoding="utf-8"))
    if manifest.get("reference_checkpoint_metadata") != checkpointmetadata:
        raise RuntimeError(f"Reference checkpoint mismatch for {track}: {cachepath}")
    preparation = manifest.get("preparation", {})
    if preparation.get("maximum_sequence_length") != maximumlength:
        print("Reference preparation:", json.dumps(preparation, indent=2)[:3500])
        raise RuntimeError(f"Reference cache length mismatch for {track}: expected {maximumlength}, got {preparation.get('maximum_sequence_length')}")
    if manifest.get("score_reduction") != "summed response-token log probabilities":
        raise RuntimeError(f"Unexpected reference score reduction for {track}: {cachepath}")
    return manifest

for track, settings in tracksettings.items():
    if not settings["enabled"]:
        continue
    preparedpath = locateexistingartifact(settings["prepared"])
    cachepath = locateexistingartifact(settings["cache"])
    if cachepath is not None:
        checkcache(cachepath, track, settings["maximumlength"])
        settings["resolvedcache"] = str(cachepath)
        print(f"{track}: using compatible existing reference cache: {cachepath}")
        continue
    if preparedpath is None:
        print(f"{track}: preparing new dataset (this requires Kaggle Internet and Hugging Face access).")
        preparepreferences(
            checkpointpath=checkpointpath,
            track=track,
            outputpath=settings["prepared"],
            trainingpairs=trainingpairs,
            validationpairs=validationpairs,
            maximumlength=settings["maximumlength"],
            maximumpromptlength=maximumpromptlength,
            seed=seed,
            datasetrevision=settings["revision"],
        )
        preparedpath = Path(settings["prepared"])
    checkprepared(preparedpath, track, settings["maximumlength"])
    settings["resolvedprepared"] = str(preparedpath)

pending = [track for track, settings in tracksettings.items() if settings["enabled"] and "resolvedcache" not in settings]
if pending:
    print("Reference cache generation needed:", pending)
    referencemodel = AutoModelForCausalLM.from_pretrained(
        str(checkpointpath), local_files_only=True, dtype=torch.float32, attn_implementation="sdpa"
    ).to("cuda:1")
    referencemodel.eval()
    referencemodel.requires_grad_(False)
    referencemodel.config.use_cache = False
    try:
        for track in pending:
            settings = tracksettings[track]
            cachereferencescores(
                referencemodel=referencemodel,
                preparedpath=settings["resolvedprepared"],
                outputpath=settings["cache"],
                batchsize=1,
            )
            settings["resolvedcache"] = settings["cache"]
            checkcache(Path(settings["resolvedcache"]), track, settings["maximumlength"])
    finally:
        del referencemodel
        gc.collect()
        torch.cuda.empty_cache()

print("Ready caches:", {track: settings.get("resolvedcache") for track, settings in tracksettings.items()})


hh: preparing new dataset (this requires Kaggle Internet and Hugging Face access).


NameError: name 'preparepreferences' is not defined

## 4. Standard DPO training functions
Original loss/scoring and accumulation logic are retained. We add periodic **non-overwriting** checkpoints and disk-space checks. Each new run starts by loading the original SFT model.

In [ ]:
import gc
import json
import math
import time
from functools import partial
from pathlib import Path

import torch

from datasets import load_from_disk
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from alignment.dpo import computepreferencelossandmetrics
from alignment.preferences import collatepreferencebatch
from alignment.scoring import compute_batch_response_statistics


def collatecachedpreferences(examples, paddingtokenid):
    preferencebatch = collatepreferencebatch(
        examples,
        paddingtokenid,
    )

    chosenscores = []
    rejectedscores = []

    for example in examples:
        chosenscores.append(example["reference_chosen_logps"])
        rejectedscores.append(example["reference_rejected_logps"])

    preferencebatch["referencechosenscores"] = torch.tensor(
        chosenscores,
        dtype=torch.float32,
    )
    preferencebatch["referencerejectedscores"] = torch.tensor(
        rejectedscores,
        dtype=torch.float32,
    )

    return preferencebatch


def computecachedpreferenceloss(model, preferencebatch, beta):
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        chosenstatistics = compute_batch_response_statistics(
            model,
            preferencebatch["chosen"],
        )
        rejectedstatistics = compute_batch_response_statistics(
            model,
            preferencebatch["rejected"],
        )

    chosenprobabilities = chosenstatistics[0]
    rejectedprobabilities = rejectedstatistics[0]

    loss, metrics = computepreferencelossandmetrics(
        chosenprobabilities,
        rejectedprobabilities,
        preferencebatch["referencechosenscores"],
        preferencebatch["referencerejectedscores"],
        beta,
    )

    device = chosenprobabilities.device

    chosendifference = (
        chosenprobabilities.detach()
        - preferencebatch["referencechosenscores"].to(device)
    ).abs().max()

    rejecteddifference = (
        rejectedprobabilities.detach()
        - preferencebatch["referencerejectedscores"].to(device)
    ).abs().max()

    metrics["maximum_reference_difference"] = torch.maximum(
        chosendifference,
        rejecteddifference,
    )

    return loss, metrics


def evaluatepreferences(model, loader, beta):
    previousmode = model.training
    model.eval()

    totals = {
        "dpo_loss": 0.0,
        "reward_accuracy": 0.0,
        "reward_margin": 0.0,
        "chosen_reward": 0.0,
        "rejected_reward": 0.0,
    }

    totalpairs = 0
    maximumdifference = 0.0

    try:
        with torch.no_grad():
            for preferencebatch in loader:
                loss, metrics = computecachedpreferenceloss(
                    model,
                    preferencebatch,
                    beta,
                )

                paircount = preferencebatch["referencechosenscores"].numel()
                totalpairs += paircount

                for metricname in totals:
                    totals[metricname] += (
                        float(metrics[metricname].item()) * paircount
                    )

                maximumdifference = max(
                    maximumdifference,
                    float(metrics["maximum_reference_difference"].item()),
                )
    finally:
        model.train(previousmode)

    if totalpairs == 0:
        raise ValueError("Validation contains no preference pairs.")

    results = {}

    for metricname, metrictotal in totals.items():
        results[metricname] = metrictotal / totalpairs

        if not math.isfinite(results[metricname]):
            raise FloatingPointError(
                f"Validation metric is not finite: {metricname}"
            )

    results["pairs"] = totalpairs
    results["maximum_reference_difference"] = maximumdifference

    return results


def appendpreferencemetrics(destination, record):
    with destination.open("a", encoding="utf-8") as outputfile:
        outputfile.write(
            json.dumps(record, allow_nan=False) + "\n"
        )


def savepreferencecheckpoint(model, tokenizer, destination, metadata):
    destination.mkdir(parents=True, exist_ok=True)

    previouscache = model.config.use_cache
    model.config.use_cache = True

    try:
        model.save_pretrained(
            str(destination),
            safe_serialization=True,
        )
        tokenizer.save_pretrained(str(destination))

        (destination / "checkpoint.json").write_text(
            json.dumps(metadata, indent=2, allow_nan=False),
            encoding="utf-8",
        )
    finally:
        model.config.use_cache = previouscache

In [ ]:
import shutil
def trainpreferences(
    cachepath,
    outputpath,
    accumulation=4,
    maxsteps=10,
    epochs=1,
    beta=0.1,
    learningrate=1e-6,
    evaluationinterval=5,
    checkpointinterval=200,
    seed=42,
):
    if accumulation < 1 or epochs < 1 or evaluationinterval < 1 or checkpointinterval < 1:
        raise ValueError("Training counts must be positive.")

    if maxsteps < 0:
        raise ValueError("maxsteps must be nonnegative.")

    if not math.isfinite(beta) or beta <= 0:
        raise ValueError("beta must be finite and positive.")

    if not math.isfinite(learningrate) or learningrate <= 0:
        raise ValueError("learningrate must be finite and positive.")

    cachepath = Path(cachepath)
    outputpath = Path(outputpath)

    if outputpath.exists():
        raise FileExistsError(
            f"Training output already exists: {outputpath}"
        )

    cachemanifest = json.loads(
        (cachepath / "manifest.json").read_text(encoding="utf-8")
    )

    if (
        cachemanifest["reference_checkpoint_metadata"]
        != checkpointmetadata
    ):
        raise ValueError("Reference cache belongs to a different checkpoint.")

    dataset = load_from_disk(str(cachepath / "dataset"))

    if len(dataset["train"]) == 0 or len(dataset["validation"]) == 0:
        raise ValueError("Training and validation datasets must not be empty.")

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    device = torch.device("cuda:0")
    torch.cuda.reset_peak_memory_stats(device)

    tokenizer = AutoTokenizer.from_pretrained(
        str(checkpointpath),
        local_files_only=True,
        fix_mistral_regex=False,
    )

    collator = partial(
        collatecachedpreferences,
        paddingtokenid=tokenizer.pad_token_id,
    )

    generator = torch.Generator().manual_seed(seed)

    trainingloader = DataLoader(
        dataset["train"],
        batch_size=1,
        shuffle=True,
        generator=generator,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
    )

    validationloader = DataLoader(
        dataset["validation"],
        batch_size=1,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
    )

    plannedsteps = math.ceil(len(trainingloader) / accumulation) * epochs

    if maxsteps:
        plannedsteps = min(plannedsteps, maxsteps)

    print("Loading trainable SFT policy.", flush=True)

    model = AutoModelForCausalLM.from_pretrained(
        str(checkpointpath),
        local_files_only=True,
        dtype=torch.float32,
        attn_implementation="sdpa",
    ).to(device)

    model.config.use_cache = False
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learningrate,
        weight_decay=0.0,
        fused=True,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        init_scale=2048.0,
        growth_interval=2000,
    )

    configuration = {
        "cache_manifest": cachemanifest,
        "policy_initialization": str(checkpointpath),
        "batch_size": 1,
        "gradient_accumulation": accumulation,
        "nominal_effective_batch_size": accumulation,
        "epochs": epochs,
        "planned_optimizer_attempts": plannedsteps,
        "beta": beta,
        "optimizer": "AdamW",
        "learning_rate": learningrate,
        "scheduler": "none",
        "weight_decay": 0.0,
        "gradient_clipping": 1.0,
        "gradient_checkpointing": True,
        "precision": "FP32 weights with FP16 autocast and GradScaler",
        "initial_gradient_scale": 2048.0,
        "scale_growth_interval": 2000,
        "seed": seed,
        "checkpoint_selection": "highest validation reward accuracy (proxy; select final generator separately)",
        "checkpoint_interval": checkpointinterval,
        "torch_version": torch.__version__,
    }

    outputpath.mkdir(parents=True, exist_ok=False)

    (outputpath / "configuration.json").write_text(
        json.dumps(configuration, indent=2),
        encoding="utf-8",
    )

    metricspath = outputpath / "metrics.jsonl"
    starttime = time.perf_counter()
    attemptedsteps = 0
    optimizersteps = 0
    skippedsteps = 0
    consecutiveskips = 0
    beststep = 0

    try:
        print("Evaluating the initial policy.", flush=True)

        validationresults = evaluatepreferences(
            model,
            validationloader,
            beta,
        )

        print(
            "Initial validation:",
            json.dumps(validationresults, indent=2),
            flush=True,
        )

        appendpreferencemetrics(
            metricspath,
            {
                "event": "validation",
                "optimizer_step": 0,
                **validationresults,
                "elapsed_seconds": time.perf_counter() - starttime,
            },
        )

        bestaccuracy = validationresults["reward_accuracy"]

        savepreferencecheckpoint(
            model,
            tokenizer,
            outputpath / "best",
            {
                "optimizer_step": 0,
                "validation": validationresults,
            },
        )

        model.train()

        with tqdm(
            total=plannedsteps,
            desc="DPO optimizer attempts",
        ) as progress:
            for epoch in range(epochs):
                iterator = iter(trainingloader)

                while attemptedsteps < plannedsteps:
                    batches = []

                    for microbatchnumber in range(accumulation):
                        preferencebatch = next(iterator, None)

                        if preferencebatch is None:
                            break

                        batches.append(preferencebatch)

                    if not batches:
                        break

                    windowpairs = sum(
                        batch["referencechosenscores"].numel()
                        for batch in batches
                    )

                    optimizer.zero_grad(set_to_none=True)
                    windowloss = 0.0

                    for preferencebatch in batches:
                        loss, metrics = computecachedpreferenceloss(
                            model,
                            preferencebatch,
                            beta,
                        )

                        paircount = (
                            preferencebatch["referencechosenscores"].numel()
                        )

                        weightedloss = loss * paircount / windowpairs
                        scaler.scale(weightedloss).backward()

                        windowloss += (
                            float(loss.detach().item())
                            * paircount / windowpairs
                        )

                    if not math.isfinite(windowloss):
                        raise FloatingPointError("DPO loss is not finite.")

                    scaler.unscale_(optimizer)

                    gradientnorm = torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=1.0,
                    )

                    gradientvalue = float(gradientnorm.item())

                    if not math.isfinite(gradientvalue):
                        gradientvalue = None

                    previousscale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()

                    updatesucceeded = scaler.get_scale() >= previousscale
                    attemptedsteps += 1

                    if updatesucceeded:
                        optimizersteps += 1
                        consecutiveskips = 0
                    else:
                        skippedsteps += 1
                        consecutiveskips += 1

                    appendpreferencemetrics(
                        metricspath,
                        {
                            "event": "training",
                            "epoch": epoch + 1,
                            "optimizer_step": optimizersteps,
                            "attempted_step": attemptedsteps,
                            "dpo_loss": windowloss,
                            "pairs": windowpairs,
                            "update_skipped": not updatesucceeded,
                            "gradient_scale_before": previousscale,
                            "gradient_scale_after": scaler.get_scale(),
                            "gradient_norm_before_clipping": gradientvalue,
                            "elapsed_seconds": (
                                time.perf_counter() - starttime
                            ),
                        },
                    )

                    if consecutiveskips >= 10:
                        raise FloatingPointError(
                            "Ten consecutive AMP updates were skipped."
                        )

                    progress.update(1)
                    progress.set_postfix(
                        loss=f"{windowloss:.4f}",
                        updates=optimizersteps,
                    )

                    shouldvalidate = (
                        updatesucceeded
                        and optimizersteps % evaluationinterval == 0
                    ) or attemptedsteps == plannedsteps

                    if updatesucceeded and optimizersteps % checkpointinterval == 0:
                        freespace = shutil.disk_usage("/kaggle/working").free / 1024**3
                        if freespace < 4:
                            raise RuntimeError(f"Only {freespace:.2f} GiB of working disk left before checkpoint save. Increase Kaggle disk or reduce checkpoint frequency.")
                        savepreferencecheckpoint(
                            model,
                            tokenizer,
                            outputpath / f"step_{optimizersteps:05d}",
                            {"optimizer_step": optimizersteps, "attempted_step": attemptedsteps},
                        )
                        print(f"Saved periodic checkpoint: step {optimizersteps}", flush=True)

                    if shouldvalidate:
                        validationresults = evaluatepreferences(
                            model,
                            validationloader,
                            beta,
                        )

                        appendpreferencemetrics(
                            metricspath,
                            {
                                "event": "validation",
                                "optimizer_step": optimizersteps,
                                **validationresults,
                                "elapsed_seconds": (
                                    time.perf_counter() - starttime
                                ),
                            },
                        )

                        if (
                            validationresults["reward_accuracy"]
                            > bestaccuracy
                        ):
                            bestaccuracy = (
                                validationresults["reward_accuracy"]
                            )
                            beststep = optimizersteps

                            savepreferencecheckpoint(
                                model,
                                tokenizer,
                                outputpath / "best",
                                {
                                    "optimizer_step": beststep,
                                    "validation": validationresults,
                                },
                            )

                if attemptedsteps >= plannedsteps:
                    break

        savepreferencecheckpoint(
            model,
            tokenizer,
            outputpath / "final",
            {
                "optimizer_step": optimizersteps,
                "validation": validationresults,
            },
        )

        summary = {
            "optimizer_steps": optimizersteps,
            "attempted_steps": attemptedsteps,
            "skipped_steps": skippedsteps,
            "best_optimizer_step": beststep,
            "best_validation_reward_accuracy": bestaccuracy,
            "final_validation": validationresults,
            "final_gradient_scale": scaler.get_scale(),
            "elapsed_seconds": time.perf_counter() - starttime,
            "peak_allocated_gpu_gib": (
                torch.cuda.max_memory_allocated(device) / 1024 ** 3
            ),
        }

        (outputpath / "summary.json").write_text(
            json.dumps(summary, indent=2, allow_nan=False),
            encoding="utf-8",
        )

        print(json.dumps(summary, indent=2), flush=True)

        return summary

    finally:
        del optimizer
        del model
        gc.collect()
        torch.cuda.empty_cache()

## 5. Run HH-RLHF and UltraFeedback
These are **two distinct runs**, each loading the original SFT policy. The default runs are full retrainings; `runhh` and `runuf` can be switched off in the configuration cell. An existing output directory raises an error rather than silently overwriting weights.

In [ ]:
results = {}

for track in ("hh", "ultrafeedback"):
    settings = tracksettings[track]
    if not settings["enabled"]:
        print(f"Skipping {track}: disabled in configuration.")
        continue

    runname = f"{track}_lr{settings['learningrate']:.0e}_ep{settings['epochs']}_len{settings['maximumlength']}"
    destination = outputroot / runname
    if destination.exists():
        raise FileExistsError(
            f"Run directory already exists: {destination}. Previous results were preserved. "
            "Choose a new outputroot or runname rather than overwriting."
        )

    print(f"\n{'='*65}\nSTARTING {track.upper()} DPO\nOutput: {destination}\n{'='*65}", flush=True)
    results[track] = trainpreferences(
        cachepath=settings["resolvedcache"],
        outputpath=str(destination),
        accumulation=accumulation,
        maxsteps=0,
        epochs=settings["epochs"],
        beta=beta,
        learningrate=settings["learningrate"],
        evaluationinterval=validationinterval,
        checkpointinterval=checkpointinterval,
        seed=seed,
    )
    print(f"COMPLETED {track.upper()}:", json.dumps(results[track], indent=2), flush=True)
    gc.collect()
    torch.cuda.empty_cache()


## 6. Training and validation loss curves
Plots use actual `metrics.jsonl` records and can be rerun if a training job was interrupted. Each run gets loss, reward accuracy, and reward margin plots, plus a CSV-like summary table.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

rundirectories = []
for metricspath in sorted(outputroot.rglob("metrics.jsonl")):
    rundirectories.append(metricspath.parent)

if not rundirectories:
    print("No metrics files found yet. Run the training cell first.")

summaryrows = []
for rundirectory in rundirectories:
    metricspath = rundirectory / "metrics.jsonl"
    records = []
    for line in metricspath.read_text(encoding="utf-8").splitlines():
        if line.strip():
            records.append(json.loads(line))

    training = [record for record in records if record.get("event") == "training"]
    validation = [record for record in records if record.get("event") == "validation"]
    if not validation:
        print("No validation metrics available:", rundirectory)
        continue

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    if training:
        axes[0].plot([row["attempted_step"] for row in training], [row["dpo_loss"] for row in training], label="Train DPO loss", alpha=0.65)
    axes[0].plot([row["optimizer_step"] for row in validation], [row["dpo_loss"] for row in validation], marker="o", label="Validation DPO loss")
    axes[0].set_title("DPO loss")
    axes[0].set_xlabel("Optimizer attempt / step")
    axes[0].legend()

    axes[1].plot([row["optimizer_step"] for row in validation], [row["reward_accuracy"] for row in validation], marker="o")
    axes[1].set_title("Validation reward accuracy")
    axes[1].set_xlabel("Optimizer step")
    axes[1].set_ylabel("Fraction")

    axes[2].plot([row["optimizer_step"] for row in validation], [row["reward_margin"] for row in validation], marker="o")
    axes[2].set_title("Validation reward margin")
    axes[2].set_xlabel("Optimizer step")

    fig.suptitle(rundirectory.name)
    fig.tight_layout()
    figurepath = rundirectory / "training_curves.png"
    fig.savefig(figurepath, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    best = max(validation, key=lambda row: row["reward_accuracy"])
    summaryrows.append({
        "run": rundirectory.name,
        "best_validation_step": best["optimizer_step"],
        "best_validation_accuracy": best["reward_accuracy"],
        "final_validation_accuracy": validation[-1]["reward_accuracy"],
        "final_validation_loss": validation[-1]["dpo_loss"],
        "training_updates_logged": len(training),
        "completed": (rundirectory / "summary.json").is_file(),
    })

if summaryrows:
    display(pd.DataFrame(summaryrows))


## 7. Optional learning-rate search (disabled by default)
**Not necessary for the first two runs.** If generation-based checkpoint comparisons show poor performance, try a *small* LR comparison instead of a large grid. Every candidate starts from the original SFT checkpoint and uses the same reference cache. This is expensive: each candidate is another full DPO training. Set `runlrsweep=True` in configuration to enable it, and ideally run it in a separate Kaggle session after saving the main outputs.

In [ ]:
if runlrsweep:
    for track in ("hh", "ultrafeedback"):
        settings = tracksettings[track]
        if not settings["enabled"]:
            continue
        for learningrate in (5e-7, 1e-6):
            runname = f"{track}_sweep_lr{learningrate:.0e}_ep{settings['epochs']}_len{settings['maximumlength']}"
            destination = outputroot / runname
            if destination.exists():
                raise FileExistsError(f"Sweep destination already exists: {destination}")
            print(f"SWEEP: {track}, LR={learningrate}", flush=True)
            trainpreferences(
                cachepath=settings["resolvedcache"],
                outputpath=str(destination),
                accumulation=accumulation,
                maxsteps=0,
                epochs=settings["epochs"],
                beta=beta,
                learningrate=learningrate,
                evaluationinterval=validationinterval,
                checkpointinterval=checkpointinterval,
                seed=seed,
            )
else:
    print("LR sweep disabled. Both main training runs remain enabled by default.")


## 8. Outputs, disk, and evaluation notes

- `outputs/dpo_retrain_2026/hh_.../` and `ultrafeedback_.../` each contain `step_00200/`, `step_00400/`, etc., `best/`, `final/`, `metrics.jsonl`, `configuration.json`, `summary.json`, and `training_curves.png`.
- With 15,000 pairs and accumulation 32, the full run is approximately **469 attempted updates per epoch**: HH ~1,407, UF ~938. AMP skipped steps may reduce successful update counts.
- **Disk use is significant**: every saved full-model checkpoint consumes space. At 200-step intervals, expect multiple gigabytes across both runs. Monitor `/kaggle/working` and save Kaggle notebook outputs when finished.
- **Selection**: `best` means highest held-out preference accuracy, not necessarily highest Gemini win rate. Use a **separate validation sample** for comparing saved checkpoint generations. Run the fixed 300-prompt HH and AlpacaEval judging sets only for final reporting, not repeated hyperparameter tuning.
- **No claim of improved win rate is made until evaluation.** Keep your earlier 46.83% HH result and UF baseline comparisons for the report.
